In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("nutriscan_products.csv")
df

,report_id,name,brand,barcode,pack_size,seller_label,product_url,data_source,analyzed_at,quality_score,...,eu_banned_count,harmful_dye_count,who_sweetener_count,controversial_additive_count,banned_in_eu_count,restricted_in_eu_count,warning_in_eu_count,permitted_in_eu_count,max_evidence_strength,ingredient_text_length
0,live-e8cd71a6-4eb7-444a-b837-b2c43883ec46,Parle-G Biscuits,Parle,8.901719e+12,40 g,JioMart,https://www.jiomart.com/product/parle-g-biscui...,gemini-estimated,2026-08-18T13:43:39.163Z,41,...,0,0,0,0,0,0,0,0,0,153
1,live-b95ab5b9-9eca-44bf-8766-d9d0d508d05d,Lay's Spanish Tomato Tango Potato Chips 58 g,Lay's,NaN,58 g,JioMart,https://www.jiomart.com/product/lay-s-spanish-...,gemini-estimated,2026-08-18T13:51:38.629Z,13,...,0,0,0,0,0,0,0,4,2,310
2,live-553d3332-0724-4827-ad03-720ded776776,Too Yumm! Indian Masala Potato Chips,Too Yumm!,NaN,82 g,JioMart,https://www.jiomart.com/product/too-yumm-india...,gemini-estimated,2026-08-18T13:52:37.133Z,17,...,0,0,0,0,0,0,0,3,2,389
3,live-09d03c2c-73b4-4223-bcfd-da3e60056350,Too Yumm Sour Cream & Onion Veggie Stix,Too Yumm,8.906093e+12,70 g,JioMart,https://www.jiomart.com/product/too-yumm-sour-...,gemini-estimated,2026-08-18T13:54:07.209Z,24,...,0,0,0,0,0,0,0,3,2,312
4,live-4de7fbe0-1c04-4efe-872a-b60cffeadbe8,Haldirams Aloo Bhujia,Haldiram,1.275111e+07,200 g,JioMart,https://www.jiomart.com/product/haldirams-nagp...,openfoodfacts,2026-08-18T14:20:02.939Z,44,...,0,0,0,0,0,0,0,4,2,642
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4333,off-8901262010016,Amul Pasteurized Butter,Amul,8.901262e+12,100 g,OpenFoodFacts,offseed://8901262010016,openfoodfacts,2026-08-31T10:04:54.516Z,0,...,0,0,0,0,0,0,0,0,0,64
4334,off-8901088155496,Saffola oats,Saffola,8.901088e+12,500 g,OpenFoodFacts,offseed://8901088155496,openfoodfacts,2026-08-31T10:09:02.999Z,100,...,0,0,0,0,0,0,0,0,0,11
4335,off-8901262140065,Lite Bread Spread,Amul,8.901262e+12,500 g,OpenFoodFacts,offseed://8901262140065,openfoodfacts,2026-08-31T10:09:03.052Z,42,...,0,0,0,0,0,0,0,0,0,178
4336,off-8901262071048,Amul Sugar free chocolate,Amul,8.901262e+12,150 g,OpenFoodFacts,offseed://8901262071048,openfoodfacts,2026-08-31T10:09:03.081Z,62,...,0,0,0,0,0,0,0,0,0,151


In [4]:
df.columns

Index(['report_id', 'name', 'brand', 'barcode', 'pack_size', 'seller_label',
       'product_url', 'data_source', 'analyzed_at', 'quality_score', 'tier',
       'nova_group', 'nova_penalty', 'efsa_penalty', 'macro_penalty',
       'trans_fat_penalty', 'trans_fat_gate_legacy', 'palm_oil_penalty',
       'positive_buffer', 'energy_kcal', 'protein_g', 'carbohydrate_g',
       'sugars_g', 'fat_g', 'saturated_fat_g', 'trans_fat_g', 'fiber_g',
       'sodium_mg', 'pack_size_grams', 'who_breach_count', 'who_breaches',
       'additive_flag_count', 'cleared_additives_count', 'eu_banned_count',
       'harmful_dye_count', 'who_sweetener_count',
       'controversial_additive_count', 'banned_in_eu_count',
       'restricted_in_eu_count', 'warning_in_eu_count',
       'permitted_in_eu_count', 'max_evidence_strength',
       'ingredient_text_length'],
      dtype='object')

In [21]:
import re

CATEGORY_KEYWORDS = [
    ("pet_food", ["dog","cat food","puppy","kitten","drools","firstbark"]),
    ("noodles", ["noodle","pasta","vermicelli","seviyan","semiya","sevai","macaroni","spaghetti","penne","hakka","ramen","chowmein","maggi"]),
    ("biscuits", ["biscuit","cookie","cracker","rusk","khari","marie","wafer","toastea","oreo","hide and seek","bourbon","nutrichoice","nutri choice","good day","krackjack","monaco","milk bikis","dark fantasy","diskette"]),
    ("bakery", ["bread","bun","cake","pastry","muffin","croissant","brownie","donut","doughnut","pav bun","toast","baguette","pizza base","spring roll","dough"]),
    ("breakfast_cereal", ["muesli","granola","cornflake","corn flake","cereal","oats","oat","porridge","daliya","dalia","poha","upma","froot loop","fruit loop","special k","chocos","kellogg"]),
    ("health_supplements", ["protein powder","protein bar","protein shake","protein isolate","energy bar","whey","chyawanprash","chyavanprash","chyawanplus","ashwagandha","shilajit","supplement","gainer","nutrition","health drink","spirulina","apple cider","horlicks","bournvita","bourn vita","boost","complan","protinex","pediasure","herbalife","malt","shake","ensure","isolate","psyllium","isabgol","giloy","gokhru","gokshura","restora","groviva","threptin","babyvita","womens plus","fiber complex","enteral"]),
    ("snacks", ["chips","chipps","namkeen","bhujia","bhujiya","mixture","sev","popcorn","makhana","khakhra","khakhara","fryums","papad","nachos","puff","mathri","chakli","murukku","snack","crisps","stix","kurkure","bhel","chiwda","chivda","khatta","farsan","chavanu","navratna","misal","lachha","french fries","aloo tikki","crunchem","simply salted","act ii","caramel","potato","ring","taco","bingo","sattu","kuch-kuch","tana-bana"]),
    ("chocolate_candy", ["chocolate","candy","toffee","lollipop","gummies","gummy","choco","marshmallow","eclairs","cocoa","dairy milk","five star","perk","munch","kitkat","gems","bar","truffle","compound","nutties","pepero"]),
    ("sweets_desserts", ["laddu","ladoo","laddoo","barfi","burfi","halwa","soan papdi","soanpapdi","gulab jamun","rasgulla","mysore pak","peda","kaju katli","jalebi","sweets","mithai","dessert","custard","jelly","ice cream","cornetto","kulfi","rabri","motichur","kala jamun","goli"]),
    ("condiments_sauces", ["sauce","peanut butter", "nut butter", "almond butter","ketchup","pickle","achar","chutney","mayonnaise","mayo","vinegar","paste","puree","dip","dressing","spread","jam","marmalade","syrup","seasoning","kissan","schezwan","conserve","gulkand","tahina","stock cube"]),
    ("dry_fruits_nuts", ["almond","badam","cashew","kaju","raisin","kishmish","pista","pistachio","walnut","akhrot","anjeer","fig","apricot","dry fruit","dryfruit","nuts","peanut","groundnut","khajur","prune","cranberry","hazelnut","dates","trail mix","nariyal","desiccated coconut"]),
    ("seeds", ["seed","chia","flax","alsi","sabja"]),
    ("spices_masala", ["masala","turmeric","turmaric","haldi","cumin","jeera","coriander","corainder","dhaniya","dhania","chilli","chili","chilly","mirch","pepper","clove","laung","cardamom","elaichi","cinnamon","dalchini","asafoetida","hing","fenugreek","methi","bay leaf","tej patta","spice","garam","sambar","rasam","kitchen king","curry powder","ajwain","saunf","fennel","nutmeg","star anise","kalonji","amchur","amchor","chaat","salt","mustard","sarso","rai","rosemary","herb"]),
    ("beverages", ["juice","drink","cola","soda","tea","coffee","squash","lemonade","smoothie","water","beverage","kombucha","milkshake","buttermilk","coconut water","thandai","paper boat","aamras","thums up"]),
    ("dairy", ["milk","curd","dahi","paneer","cheese","cheddar","butter","yogurt","yoghurt","lassi","cream","khoya","condensed","tofu"]),
    ("oils_fats", ["oil","ghee","vanaspati","margarine"]),
    ("sweeteners", ["sugar","jaggery","gur","honey","stevia","sweetener","misri"]),
    ("pulses_legumes", ["dal","dahl","daal","chana","channa","rajma","moong","masoor","toor","tur","urad","lobia","chickpea","kabuli","matar","pea","lentil","soybean","soya","bean","gram"]),
    ("staples_grains", ["atta","aata","flour","maida","rice","besan","sooji","suji","rava","ravva","quinoa","millet","ragi","bajra","jowar","wheat","barley","jau","corn","makai","sabudana","idli","idly","dosa","dosai","batter","india gate","rozana","grain","sushi"]),
    ("ready_to_eat", ["ready to eat","ready-to-eat","instant","mix","gravy","curry","biryani","pulao","soup","dhokla","khaman","bhaji","paratha","roti","tikka","momo","uttappam","mock meat","mock chicken","chicken"]),
    ("fresh_produce", ["onion","tomato","vegetable","fruit","banana","apple","mango","lemon","nimbu","garlic","ginger","carrot","spinach","capsicum","lauki"]),
    ("gift_combo", ["gift","combo","hamper","pack of","assorted","pcs","box"]),
]

def build(keywords):
    # words of 6+ letters: match any ending  ("biscuit" also catches "biscuits")
    # short words: must be the whole word    ("sev" must NOT catch "seviyan")
    parts = [re.escape(k) + (r"\w*" if len(k) >= 6 else r"e?s?\b") for k in keywords]
    return re.compile(r"\b(?:" + "|".join(parts) + r")", re.I)

PATTERNS = [(cat, build(kws)) for cat, kws in CATEGORY_KEYWORDS]

def find_category(name):
    n = str(name).lower()
    for cat, pattern in PATTERNS:
        if pattern.search(n):      # first match wins — order matters!
            return cat
    return "others"


In [22]:
df["category"] = df["name"].apply(find_category)
df["category"].value_counts()

C:\Users\91912\AppData\Local\Temp\ipykernel_24028\2557876140.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["category"] = df["name"].apply(find_category)


category
snacks                445
biscuits              442
breakfast_cereal      343
staples_grains        328
pulses_legumes        263
spices_masala         258
noodles               256
condiments_sauces     231
health_supplements    205
chocolate_candy       182
dairy                 169
dry_fruits_nuts       147
beverages              84
seeds                  72
sweets_desserts        69
bakery                 62
ready_to_eat           32
sweeteners             25
fresh_produce          15
others                 11
gift_combo              6
oils_fats               4
Name: count, dtype: int64

In [7]:
sample = df.sample(100, random_state = 42)[["name", "category"]]

In [8]:
sample.to_csv("category_check.csv", index = False)

In [22]:
temp = pd.read_csv("category_check.csv")

In [24]:
temp["Marking"].value_counts()

Marking
Right    95
Wrong     5
Name: count, dtype: int64

In [13]:
df = df[df["category"] != "pet_food"]

In [16]:
df = df.drop_duplicates(subset = ["name", "brand"])

In [17]:
df["is_bad"] = (df["quality_score"] < 50).astype(int)

C:\Users\91912\AppData\Local\Temp\ipykernel_24028\3581458455.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["is_bad"] = (df["quality_score"] < 50).astype(int)


In [18]:
print(df.shape)

(3649, 45)


In [23]:
already_checked = pd.read_csv("category_check.csv")["name"]
fresh = df[~df["name"].isin(already_checked)]
sample2 = fresh.sample(50, random_state = 99)[["name", "category"]]
sample2.to_csv("category_check_v2.csv", index = False)

In [27]:
temp = pd.read_csv("category_check_v2.csv")

In [28]:
temp["mark"].value_counts()

mark
right    50
Name: count, dtype: int64